# 変数を組み合わせて特徴を見つける
既存の列を組み合わせたり、文字列から情報を取り出したりして、生存と関係する新しい切り口を探します。分析で見つけたことを整理し、次の講座(前処理と特徴量の作成)につなげます。

## 学習目標
- 複数の列を計算で組み合わせて、新しい列(家族の人数など)を作れる
- 文字列の列から、正規表現で必要な部分(敬称)を取り出せる
- 「生存率が高すぎる」列を見つけたときに、リークを疑える
- 分析で見つけたことを、特徴量のアイデアとして整理できる

## 家族の人数

`SibSp`(兄弟姉妹・配偶者)と `Parch`(親・子)を足し、本人の 1 人を加えると、一緒に乗船した家族の人数になります。

In [1]:
import pandas as pd

# 学習用のデータを読み込む
train = pd.read_csv("train.csv")

# 新しい列 FamilySize を作る(列どうしの足し算は、行ごとに計算される)
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1

# 家族の人数ごとの生存率と人数を求める
train.groupby("FamilySize")["Survived"].agg(["mean", "count"]).round(3)

             mean  count
FamilySize              
1           0.302    547
2           0.520    152
3           0.578    109
4           0.652     23
5           0.294     17
6           0.176     17
7           0.231     13
8           0.000      4
11          0.000      9


1 人で乗船した乗客(547 人)の生存率は 30% ですが、2〜4 人の家族では 52〜65% に上がり、5 人以上では再び下がります。`SibSp`・`Parch` を別々に見るより、合計した方が傾向がはっきりします。ただし、人数が 5 人以上のグループはそれぞれの人数が少ないので、いくつかのグループにまとめて扱うのがよさそうです(演習で試します)。

## 名前から敬称を取り出す

名前は「姓, 敬称. 名」の形で書かれています。`str.extract` に **正規表現** を渡すと、かっこ `( )` で囲んだ部分を取り出せます。`,\s*([^.]+)\.` は「カンマと空白の後ろから、ピリオドの前までの文字」を表します。

In [2]:
# 正規表現の意味: ","(カンマ)→ "\s*"(空白 0 文字以上)→ "([^.]+)"(ピリオド以外の文字の並び。かっこの中が取り出す部分)→ "\."(ピリオド)
# str.extract は取り出した結果を表で返すので、[0] で 1 列目を取り出して Title 列にする
train["Title"] = train["Name"].str.extract(r",\s*([^.]+)\.")[0]

# 名前と、取り出した敬称を並べて確認する
train[["Name", "Title"]].head(4)

                                              Name Title
0  Bucknell, Mrs. William Robert (Emma Eliza Ward)   Mrs
1                   Saundercock, Mr. William Henry    Mr
2      Renouf, Mrs. Peter Henry (Lillian Jefferys)   Mrs
3                           Augustsson, Mr. Albert    Mr


In [3]:
# 敬称ごとの人数を、多い順に 6 つ表示する
train["Title"].value_counts().head(6)

Title
Mr        525
Miss      164
Mrs       134
Master     41
Rev         7
Dr          6
Name: count, dtype: int64


敬称には、性別だけでなく年齢や立場の情報も含まれています。`Master` は男の子の敬称、`Miss` は未婚の女性(女の子を含む)の敬称です。年齢が欠損していても、敬称から大まかな年齢層を推測できます(次の講座で、年齢の補完に使います)。

## 「良すぎる」関係はリークを疑う

救命ボートの番号 `Boat` が記録されているかどうかで、生存率を比べてみます。

In [4]:
# 救命ボートの番号が記録されているか(True / False)で分けて、生存率を求める
has_boat = train["Boat"].notna()
train.groupby(has_boat)["Survived"].mean().round(3)

Boat
False    0.027
True     0.978
Name: Survived, dtype: float64


ボートの記録がある乗客の生存率は 98%、ない乗客は 3% と、ほぼ完全に生存を言い当てています。このように **生存率がほぼ 0 か 1 に分かれる列** を見つけたら、事故の後に決まった情報(リーク)ではないかを疑いましょう。`Boat` は予測用のデータにはなく、予測には使えません。

## 分析で見つけたことの整理

| 見つけたこと | 次の講座で作る特徴量のアイデア |
|---|---|
| 女性の生存率が非常に高い | `Sex` を数値(0 / 1)に変換する |
| 等級が高いほど生存率が高い | `Pclass` をそのまま、または等級ごとの列に分けて使う |
| 子どもは生存しやすい | 年齢を補完して使う(敬称から推測)、子どもかどうかの列を作る |
| 運賃は右に裾が長い | 対数変換して使う |
| 家族の人数で生存率が変わる | 家族の人数・1 人かどうかの列を作る |
| 客室番号の有無で生存率が違う | 客室番号があるかどうかの列を作る |
| `Boat`・`Body` は生存をほぼ言い当てる | **使わない**(リーク) |

## 演習

学習用のデータで、家族の人数 `FamilySize`(`SibSp + Parch + 1`)を求め、区切り `[0, 1, 4, 11]` で「alone(1 人)」「small(2〜4 人)」「large(5 人以上)」の 3 グループに分けてください。グループごとの生存率(`mean`)と人数(`count`)を小数第3位まで表示してください。

In [ ]:
# @starter
import pandas as pd

# 学習用のデータを読み込む
train = pd.read_csv("train.csv")

# 家族の人数の列を作る(____ に、SibSp と Parch と本人の 1 人を足す式を書く)
train["FamilySize"] = ____

# 家族の人数を 3 グループに分ける。区間 (0, 1] → alone、(1, 4] → small、(4, 11] → large
train["FamilyGroup"] = pd.cut(train["FamilySize"], bins=[0, 1, 4, 11], labels=["alone", "small", "large"])

# グループごとの生存率と人数を求め、小数第3位まで表示する
result = train.groupby("FamilyGroup", observed=True)["Survived"].agg(["mean", "count"])
print(result.round(3))

In [5]:
# @solution
# @title: 家族の人数のグループごとの生存率
# @hint: 家族の人数は train["SibSp"] + train["Parch"] + 1 です。pd.cut の labels で区間に名前を付けられます。
# @explanation: 2〜4 人の家族(small)の生存率は約 55% で、1 人(約 30%)や 5 人以上(約 18%)より高くなっています。人数の少ないグループをまとめたことで、傾向が読み取りやすくなりました。
import pandas as pd

# 学習用のデータを読み込む
train = pd.read_csv("train.csv")

# 家族の人数 = 兄弟姉妹・配偶者 + 親・子 + 本人
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1

# 家族の人数を 3 グループに分ける。区間 (0, 1] → alone、(1, 4] → small、(4, 11] → large
train["FamilyGroup"] = pd.cut(train["FamilySize"], bins=[0, 1, 4, 11], labels=["alone", "small", "large"])

# グループごとの生存率と人数を求め、小数第3位まで表示する
result = train.groupby("FamilyGroup", observed=True)["Survived"].agg(["mean", "count"])
print(result.round(3))

              mean  count
FamilyGroup              
alone        0.302    547
small        0.553    284
large        0.183     60


学習用のデータで、名前 `Name` から敬称を取り出して `Title` 列を作り、敬称ごとの生存率(`mean`)と人数(`count`)を求めてください。**人数の多い順に上位 4 つ** を、小数第3位まで表示してください。

In [ ]:
# @starter
import pandas as pd

# 学習用のデータを読み込む
train = pd.read_csv("train.csv")

# 名前から敬称を取り出して Title 列を作る
# (____ に、正規表現のかっこの部分を取り出す str のメソッドの名前を入れる)
train["Title"] = train["Name"].str.____(r",\s*([^.]+)\.")[0]

# 敬称ごとの生存率と人数を求める
result = train.groupby("Title")["Survived"].agg(["mean", "count"])

# 人数(count)の多い順に並べ、上位 4 行を小数第3位まで表示する
# (____ に、先頭から指定した行数を取り出すメソッドの名前を入れる)
print(result.sort_values("count", ascending=False).____(4).round(3))

In [6]:
# @solution
# @title: 敬称ごとの生存率
# @hint: 正規表現で取り出すのは str.extract、上位の数行を取り出すのは head です。
# @explanation: Mr(成人男性)の生存率は約 16% と低く、Mrs(既婚女性)は約 81%、Miss(未婚女性)は約 67% です。Master(男の子)は約 46% で、成人男性よりずっと高く、子どもが優先されたことがわかります。
import pandas as pd

# 学習用のデータを読み込む
train = pd.read_csv("train.csv")

# 名前から敬称を取り出して Title 列を作る
train["Title"] = train["Name"].str.extract(r",\s*([^.]+)\.")[0]

# 敬称ごとの生存率と人数を求める
result = train.groupby("Title")["Survived"].agg(["mean", "count"])

# 人数(count)の多い順に並べ、上位 4 行を小数第3位まで表示する
print(result.sort_values("count", ascending=False).head(4).round(3))

         mean  count
Title               
Mr      0.164    525
Miss    0.671    164
Mrs     0.806    134
Master  0.463     41


<!-- @question
type: choice
answer: I と II のみ
title: 分析結果の整理
hint: 各記述が、この講座で確かめた結果と合っているか、また予測に使える情報かを確認します。
explanation: I. 敬称 Master(男の子)の生存率は約 46% で、Mr(約 16%)より高いので正しい。II. 家族の人数が 2〜4 人の乗客は、1 人の乗客より生存率が高いので正しい。III. Boat は事故の後に決まる情報(リーク)で、予測用のデータにもないため、特徴量として使ってはいけないので誤り。
-->
この講座の分析で見つけたことについて述べた次の記述 I〜III のうち、正しいものの組み合わせを選べ。

I. 男の子(敬称 Master)は、成人男性(敬称 Mr)より生存率が高い。

II. 家族の人数が 2〜4 人の乗客は、1 人で乗船した乗客より生存率が高い。

III. 救命ボートの番号 `Boat` は生存をよく言い当てるので、モデルの特徴量として使うとよい。

- I のみ
- II のみ
- I と II のみ
- I と III のみ
- I と II と III

## まとめ

- 列を組み合わせると(家族の人数 = `SibSp + Parch + 1`)、傾向がはっきりすることがある。人数の少ないグループはまとめる。
- `str.extract` と正規表現で、名前から敬称を取り出せる。敬称は性別・年齢層の情報を持つ。
- 生存をほぼ言い当てる列はリークを疑う。分析で見つけたことを、次の講座の特徴量のアイデアとして整理する。